In [1]:
# ======================
# 1. Install + Imports
# ======================
!pip install -q transformers datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 79.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour 

In [2]:
import pandas as pd
from PIL import Image
from datasets import Dataset
from transformers import BlipProcessor, BlipForConditionalGeneration, TrainingArguments, Trainer

# ======================
# 2. Load CSV Dataset
# ======================
csv_path = "/kaggle/input/hp-all/hp_all_cap.csv"  # your dataset
df = pd.read_csv(csv_path)

assert "image_path" in df.columns and "caption" in df.columns, "CSV must have 'image_path' and 'caption' columns"
dataset = Dataset.from_pandas(df)

# ======================
# 3. Choose BLIP Variant
# ======================
model_id = "Salesforce/blip-image-captioning-base"   # use "base" or "large"
processor = BlipProcessor.from_pretrained(model_id)
model = BlipForConditionalGeneration.from_pretrained(model_id).to("cuda")

# ======================
# 4. Preprocess Function
# ======================
def preprocess(batch):
    images, texts = [], []
    for img_path, caption in zip(batch["image_path"], batch["caption"]):
        try:
            images.append(Image.open(img_path).convert("RGB"))
            texts.append(caption)
        except Exception as e:
            print("Skipping:", img_path, e)
    inputs = processor(images=images, text=texts, padding="max_length",
                       truncation=True, max_length=30, return_tensors="pt")
    inputs["labels"] = inputs["input_ids"]
    return inputs

processed_dataset = dataset.map(preprocess, batched=True, remove_columns=dataset.column_names)

# ======================
# 5. Training Setup (NO CHECKPOINT SAVING)
# ======================
training_args = TrainingArguments(
    output_dir=f"/kaggle/working/blip-finetuned-hp",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    fp16=True,
    save_strategy="no",       # ⛔ No intermediate checkpoints
    logging_steps=100,
    remove_unused_columns=False,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=processed_dataset,
)

# ======================
# 6. Train + Save
# ======================
trainer.train()
model.save_pretrained(f"/kaggle/working/{model_id.split('/')[-1]}-finetuned-hp")
processor.save_pretrained(f"/kaggle/working/{model_id.split('/')[-1]}-finetuned-hp")

print(f"✅ Fine-tuned {model_id} saved without intermediate checkpoints!")


2025-09-13 17:54:10.380202: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1757786050.583330      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1757786050.644015      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Map:   0%|          | 0/1281 [00:00<?, ? examples/s]

Skipping: /kaggle/input/hp-all/HP_ALL_complete/HP_ALL/HP4/gobletoffire104.jpg [Errno 2] No such file or directory: '/kaggle/input/hp-all/HP_ALL_complete/HP_ALL/HP4/gobletoffire104.jpg'
Skipping: /kaggle/input/hp-all/HP_ALL_complete/HP_ALL/HP7/deathlyhallowsone154.jpg [Errno 2] No such file or directory: '/kaggle/input/hp-all/HP_ALL_complete/HP_ALL/HP7/deathlyhallowsone154.jpg'


Step,Training Loss
100,4.036900
200,1.022100
300,0.714800
400,0.427400


✅ Fine-tuned Salesforce/blip-image-captioning-base saved without intermediate checkpoints!


In [3]:
# # ============================================================
# # 1. Install Dependencies (Kaggle)
# # ============================================================
# !pip install -q transformers datasets accelerate

In [4]:
# import pandas as pd
# from PIL import Image
# from datasets import Dataset
# import torch
# from transformers import Blip2Processor, Blip2ForConditionalGeneration, TrainingArguments, Trainer

# # ======================
# # 1. Load Dataset
# # ======================
# csv_path = "/kaggle/input/hp-all/hp_all_cap.csv"  # Update with your file path
# df = pd.read_csv(csv_path)
# assert "image_path" in df.columns and "caption" in df.columns, "CSV must have 'image_path' and 'caption' columns"
# dataset = Dataset.from_pandas(df)

# # ======================
# # 2. Load BLIP-2 Model + Processor
# # ======================
# model_id = "Salesforce/blip2-opt-2.7b"
# processor = Blip2Processor.from_pretrained(model_id)
# model = Blip2ForConditionalGeneration.from_pretrained(model_id, torch_dtype=torch.float16).to("cuda")

# # Freeze everything except Q-Former
# for name, param in model.named_parameters():
#     if not name.startswith("qformer"):
#         param.requires_grad = False

# # ======================
# # 3. Preprocessing
# # ======================
# def preprocess(batch):
#     images, texts = [], []
#     for img_path, caption in zip(batch["image_path"], batch["caption"]):
#         try:
#             images.append(Image.open(img_path).convert("RGB"))
#             texts.append(caption)
#         except Exception as e:
#             print(f"Skipping {img_path} due to error: {e}")
#     inputs = processor(images=images, text=texts, padding="max_length",
#                        truncation=True, max_length=30, return_tensors="pt")
#     inputs["labels"] = inputs["input_ids"]
#     return inputs

# processed_dataset = dataset.map(preprocess, batched=True, remove_columns=dataset.column_names)
# processed_dataset.set_format(type="torch")

# # ======================
# # 4. Training Arguments
# # ======================
# training_args = TrainingArguments(
#     output_dir="/kaggle/working/blip2-opt-2.7b-finetuned",
#     per_device_train_batch_size=1,  # keep memory safe
#     gradient_accumulation_steps=16,  # effectively increases batch size
#     learning_rate=1e-4,
#     num_train_epochs=3,
#     fp16=True,
#     save_strategy="no",
#     logging_steps=50,
#     remove_unused_columns=False,
#     report_to=[],
# )

# trainer = Trainer(
#     model=model,
#     args=training_args,
#     train_dataset=processed_dataset,
# )

# # ======================
# # 5. Train & Save
# # ======================
# trainer.train()

# model.save_pretrained("/kaggle/working/blip2-opt-2.7b-finetuned")
# processor.save_pretrained("/kaggle/working/blip2-opt-2.7b-finetuned")

# print("✅ BLIP-2 OPT-2.7B fine-tuning complete & saved!")
